# Part 3 – Machine Learning and AI

## Task 1 – Supervised Machine Learning Models

### Objective

The objective of this task is to develop and evaluate supervised machine learning models using the processed traffic dataset created in Part 2.

Two supervised learning problems are addressed:

1. **Classification** – estimate accident likelihood using the documented proxy `high_risk` label because a real accident dataset was not provided.
2. **Regression** – predict `traffic_volume` using time, weather and holiday-related features.

For each problem, two algorithms are compared:

**Classification**
- Logistic Regression
- Random Forest Classifier

**Regression**
- Linear Regression
- Random Forest Regressor

Classification models are evaluated using accuracy, precision, recall, F1-score and ROC AUC.

Regression models are evaluated using MAE and R-squared.

A common engineered feature set is used where appropriate, including time-based features, weather encodings, a holiday flag and cyclical encodings of hour and day of week.

The `high_risk` target is a proxy label created only to demonstrate the machine learning classification workflow. It must not be interpreted as a prediction of actual traffic accidents.


In [12]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import (
    LogisticRegression,
    LinearRegression
)

from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    mean_absolute_error,
    r2_score
)


# ------------------------------------------------------------
# Logging configuration
# ------------------------------------------------------------

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
logger.propagate = False

if logger.handlers:
    logger.handlers.clear()

formatter = logging.Formatter(
    "%(asctime)s - %(levelname)s - %(name)s - %(message)s"
)

console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(formatter)

file_handler = logging.FileHandler(
    "task1_supervised_ml.log",
    mode="w"
)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)

logger.info("Task 1 supervised machine learning workflow started.")

2026-10-07 20:07:22,745 - INFO - __main__ - Task 1 supervised machine learning workflow started.


## 1.3 Load the Engineered Traffic Dataset

The engineered dataset produced in Part 2 is reused as the starting point for supervised machine learning.

The dataset is loaded from the Part 2 directory and validated before model preparation begins.

In [13]:
# ------------------------------------------------------------
# Dataset path
# ------------------------------------------------------------

DATA_PATH = Path(
    "../part2_python/engineered_traffic_data.csv"
)


# ------------------------------------------------------------
# Load dataset
# ------------------------------------------------------------

try:
    df = pd.read_csv(
        DATA_PATH,
        parse_dates=["date_time"]
    )

    logger.info(
        "Engineered traffic dataset loaded successfully: "
        "%d rows, %d columns.",
        df.shape[0],
        df.shape[1]
    )

except FileNotFoundError:
    logger.error(
        "Engineered traffic dataset was not found at: %s",
        DATA_PATH
    )
    raise

except (pd.errors.ParserError, ValueError) as error:
    logger.error(
        "Unable to load engineered traffic dataset: %s",
        error
    )
    raise

2026-10-07 20:07:23,173 - INFO - __main__ - Engineered traffic dataset loaded successfully: 48187 rows, 30 columns.


In [15]:
df.columns.tolist()

['holiday',
 'temp',
 'rain_1h',
 'snow_1h',
 'clouds_all',
 'weather_main',
 'weather_description',
 'date_time',
 'traffic_volume',
 'hour',
 'day_of_week',
 'is_weekend',
 'hour_sin',
 'hour_cos',
 'has_rain',
 'has_snow',
 'weather_Clear',
 'weather_Clouds',
 'weather_Drizzle',
 'weather_Fog',
 'weather_Haze',
 'weather_Mist',
 'weather_Rain',
 'weather_Smoke',
 'weather_Snow',
 'weather_Squall',
 'weather_Thunderstorm',
 'temp_scaled',
 'clouds_all_scaled',
 'congestion_category']

## 1.4 Prepare Additional Features Required for Machine Learning

The engineered dataset produced in Part 2 is reused as the starting point for Part 3 rather than rebuilding the complete data-processing workflow.

The dataset already contains time-based features, weather encodings, weekend indicators and cyclical hour features.

For Part 3, the following additional features are created to satisfy the supervised machine learning requirements:

- `is_holiday` – binary holiday indicator
- `day_sin` and `day_cos` – cyclical encoding of day of week
- `is_low_visibility` – indicator for low-visibility weather conditions

The existing Part 2 `congestion_category` is retained temporarily for comparison. Because Part 3 explicitly requires quartile-based congestion categories, it is subsequently rebuilt using the Part 3 quartile definition before the proxy `high_risk` classification target is created.

This approach preserves the engineered features developed in Part 2 while ensuring that the Part 3 classification workflow follows the proxy-label definition specified in the assignment.


In [16]:
# ------------------------------------------------------------
# Task 1.4 - Prepare additional Part 3 features
# ------------------------------------------------------------

required_existing_columns = [
    "holiday",
    "traffic_volume",
    "weather_main",
    "hour",
    "day_of_week",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "congestion_category"
]

missing_columns = [
    column
    for column in required_existing_columns
    if column not in df.columns
]

if missing_columns:
    logger.error(
        "Required Part 2 columns are missing: %s",
        missing_columns
    )
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

logger.info(
    "Required Part 2 engineered features validated successfully."
)


# ------------------------------------------------------------
# Holiday flag
# ------------------------------------------------------------

holiday_text = (
    df["holiday"]
    .fillna("None")
    .astype(str)
    .str.strip()
    .str.lower()
)

df["is_holiday"] = (
    ~holiday_text.isin(
        ["none", "nan", ""]
    )
).astype(int)


# ------------------------------------------------------------
# Cyclical day-of-week features
# ------------------------------------------------------------

df["day_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7
)

df["day_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7
)


# ------------------------------------------------------------
# Low-visibility indicator
# ------------------------------------------------------------

LOW_VISIBILITY_WEATHER = {
    "Fog",
    "Mist",
    "Haze",
    "Smoke"
}

df["is_low_visibility"] = (
    df["weather_main"]
    .isin(LOW_VISIBILITY_WEATHER)
    .astype(int)
)


logger.info(
    "Part 3 features created: "
    "is_holiday, day_sin, day_cos, is_low_visibility."
)

2026-10-07 20:07:33,106 - INFO - __main__ - Required Part 2 engineered features validated successfully.
2026-10-07 20:07:33,166 - INFO - __main__ - Part 3 features created: is_holiday, day_sin, day_cos, is_low_visibility.


## 1.5 Rebuild Congestion Category and Create Proxy Accident-Risk Label

The Part 2 engineered dataset contains an existing `congestion_category`. However, Part 2 used fixed traffic-volume thresholds, while the Part 3 assignment explicitly requires congestion categories based on quartiles of `traffic_volume`.

Therefore, the Part 2 category is first compared with the Part 3 quartile definition for validation. The Part 3 `congestion_category` is then rebuilt using the required quartile thresholds.

The quartile-based categories are:

- Low: traffic volume at or below Q1
- Medium: traffic volume above Q1 and at or below Q2
- High: traffic volume above Q2 and at or below Q3
- Severe: traffic volume above Q3

Because no real accident dataset was provided, a proxy `high_risk` classification target is then created from this rebuilt Part 3 congestion category.

A record is classified as high risk when High or Severe congestion occurs together with severe or low-visibility weather conditions.

This proxy label is used only to demonstrate the machine learning classification workflow and must not be interpreted as a prediction of actual traffic accidents.

In [17]:
# ------------------------------------------------------------
# Task 1.5 - Rebuild Part 3 congestion category
# and create proxy accident-risk label
# ------------------------------------------------------------

# Preserve the Part 2 category for comparison
df["congestion_category_part2"] = (
    df["congestion_category"].copy()
)


# ------------------------------------------------------------
# Calculate Part 3 quartile thresholds
# ------------------------------------------------------------

q1, q2, q3 = df["traffic_volume"].quantile(
    [0.25, 0.50, 0.75]
).values


def quartile_congestion(value):
    """Assign congestion using the Part 3 quartile definition."""

    if value <= q1:
        return "Low"
    elif value <= q2:
        return "Medium"
    elif value <= q3:
        return "High"
    else:
        return "Severe"


part3_congestion = (
    df["traffic_volume"]
    .apply(quartile_congestion)
)


# ------------------------------------------------------------
# Compare Part 2 and Part 3 definitions
# ------------------------------------------------------------

congestion_matches = (
    df["congestion_category_part2"].astype(str)
    == part3_congestion.astype(str)
)

match_percentage = (
    congestion_matches.mean() * 100
)

mismatch_count = int(
    (~congestion_matches).sum()
)


logger.info(
    "Part 2 and Part 3 congestion-category agreement: %.2f%%.",
    match_percentage
)

if mismatch_count > 0:
    logger.warning(
        "Part 2 congestion_category differs from the "
        "Part 3 quartile definition for %d records.",
        mismatch_count
    )


# ------------------------------------------------------------
# Replace with the required Part 3 definition
# ------------------------------------------------------------

df["congestion_category"] = (
    part3_congestion
)

logger.info(
    "congestion_category rebuilt using Part 3 quartiles: "
    "Q1=%.2f, Q2=%.2f, Q3=%.2f.",
    q1,
    q2,
    q3
)


# ------------------------------------------------------------
# Severe-weather definition
# ------------------------------------------------------------

SEVERE_WEATHER = {
    "Thunderstorm",
    "Snow",
    "Squall"
}


# ------------------------------------------------------------
# Create proxy accident-risk label
# ------------------------------------------------------------

high_congestion = (
    df["congestion_category"]
    .isin(["High", "Severe"])
)

risky_weather = (
    df["weather_main"].isin(SEVERE_WEATHER)
    | (df["is_low_visibility"] == 1)
)

df["high_risk"] = (
    high_congestion
    & risky_weather
).astype(int)


logger.info(
    "Proxy high_risk target rebuilt successfully "
    "using the Part 3 quartile congestion category."
)


# ------------------------------------------------------------
# User-facing validation output
# ------------------------------------------------------------

print(
    f"Part 2 vs Part 3 congestion agreement: "
    f"{match_percentage:.2f}%"
)

print(
    f"Number of different records: "
    f"{mismatch_count:,}"
)

print(
    "\nPart 3 quartile thresholds:"
)

print(f"Q1: {q1:,.2f}")
print(f"Q2: {q2:,.2f}")
print(f"Q3: {q3:,.2f}")

print(
    "\nPart 3 congestion-category counts:"
)

print(
    df["congestion_category"]
    .value_counts()
    .reindex(
        ["Low", "Medium", "High", "Severe"]
    )
)

print(
    "\nProxy high-risk class counts:"
)

print(
    df["high_risk"]
    .value_counts()
    .sort_index()
)

print(
    "\nProxy high-risk class percentages:"
)

print(
    (
        df["high_risk"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
)


2026-10-07 20:07:35,491 - INFO - __main__ - Part 2 and Part 3 congestion-category agreement: 50.01%.
2026-10-07 20:07:35,493 - WARNING - __main__ - Part 2 congestion_category differs from the Part 3 quartile definition for 24091 records.
2026-10-07 20:07:35,498 - INFO - __main__ - congestion_category rebuilt using Part 3 quartiles: Q1=1192.50, Q2=3379.00, Q3=4933.00.
2026-10-07 20:07:35,511 - INFO - __main__ - Proxy high_risk target rebuilt successfully using the Part 3 quartile congestion category.


Part 2 vs Part 3 congestion agreement: 50.01%
Number of different records: 24,091

Part 3 quartile thresholds:
Q1: 1,192.50
Q2: 3,379.00
Q3: 4,933.00

Part 3 congestion-category counts:
congestion_category
Low       12047
Medium    12049
High      12054
Severe    12037
Name: count, dtype: int64

Proxy high-risk class counts:
high_risk
0    42749
1     5438
Name: count, dtype: int64

Proxy high-risk class percentages:
high_risk
0    88.71
1    11.29
Name: proportion, dtype: float64


In [18]:
print(
    f"Congestion-category agreement: "
    f"{match_percentage:.2f}%"
)

print("\nPart 3 additional features:")
print(
    df[
        [
            "is_holiday",
            "day_sin",
            "day_cos",
            "is_low_visibility",
            "high_risk"
        ]
    ].head()
)

print("\nProxy high-risk class counts:")
print(
    df["high_risk"]
    .value_counts()
    .sort_index()
)

print("\nProxy high-risk class percentages:")
print(
    (
        df["high_risk"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
)

print("\nMissing values in Part 3 features:")
print(
    df[
        [
            "is_holiday",
            "day_sin",
            "day_cos",
            "is_low_visibility",
            "high_risk"
        ]
    ].isnull().sum()
)

Congestion-category agreement: 50.01%

Part 3 additional features:
   is_holiday   day_sin  day_cos  is_low_visibility  high_risk
0           0  0.781831  0.62349                  0          0
1           0  0.781831  0.62349                  0          0
2           0  0.781831  0.62349                  0          0
3           0  0.781831  0.62349                  0          0
4           0  0.781831  0.62349                  0          0

Proxy high-risk class counts:
high_risk
0    42749
1     5438
Name: count, dtype: int64

Proxy high-risk class percentages:
high_risk
0    88.71
1    11.29
Name: proportion, dtype: float64

Missing values in Part 3 features:
is_holiday           0
day_sin              0
day_cos              0
is_low_visibility    0
high_risk            0
dtype: int64


## 1.6 Define the Common Machine Learning Feature Set

A common engineered feature set is used for the supervised learning tasks.

The feature set contains:

- Time-based features
- Cyclical hour encoding
- Cyclical day-of-week encoding
- Weekend indicator
- Holiday indicator
- Weather measurements
- Rain and snow indicators
- One-hot encoded weather categories
- Low-visibility indicator

`traffic_volume` is excluded from the classification predictors because the proxy `high_risk` target is partly derived from congestion, which itself is calculated from traffic volume.

`congestion_category` is also excluded from the predictors because it is directly used to construct the proxy classification target. Including either variable would create target leakage.

For regression, `traffic_volume` is used only as the target variable.

In [19]:
# ------------------------------------------------------------
# Common feature set
# ------------------------------------------------------------

weather_features = [
    column
    for column in df.columns
    if column.startswith("weather_")
    and column not in [
        "weather_main",
        "weather_description"
    ]
]

base_features = [
    "hour",
    "day_of_week",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "is_holiday",
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "has_rain",
    "has_snow",
    "is_low_visibility"
]

feature_columns = []

for column in base_features + weather_features:
    if (
        column in df.columns
        and column not in feature_columns
    ):
        feature_columns.append(column)


logger.info(
    "Common ML feature set prepared with %d features.",
    len(feature_columns)
)

print("Number of ML features:", len(feature_columns))
print("\nFeatures used:")
print(feature_columns)

2026-10-07 20:11:48,253 - INFO - __main__ - Common ML feature set prepared with 26 features.


Number of ML features: 26

Features used:
['hour', 'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'is_holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'has_rain', 'has_snow', 'is_low_visibility', 'weather_Clear', 'weather_Clouds', 'weather_Drizzle', 'weather_Fog', 'weather_Haze', 'weather_Mist', 'weather_Rain', 'weather_Smoke', 'weather_Snow', 'weather_Squall', 'weather_Thunderstorm']


## 1.7 Prepare Training and Test Data

The dataset is divided into training and test sets.

For classification, a stratified split is used so that the proportion of proxy high-risk observations is maintained in both the training and test datasets.

A fixed `random_state` is used to make the experiment reproducible.

Feature scaling is applied for Logistic Regression. The Random Forest model uses the original feature values.

In [20]:
# ------------------------------------------------------------
# Classification dataset
# ------------------------------------------------------------

X_class = df[feature_columns].copy()
y_class = df["high_risk"].copy()


# ------------------------------------------------------------
# Train/test split
# ------------------------------------------------------------

(
    X_class_train,
    X_class_test,
    y_class_train,
    y_class_test
) = train_test_split(
    X_class,
    y_class,
    test_size=0.20,
    random_state=42,
    stratify=y_class
)


# ------------------------------------------------------------
# Scaling for Logistic Regression
# ------------------------------------------------------------

class_scaler = StandardScaler()

X_class_train_scaled = class_scaler.fit_transform(
    X_class_train
)

X_class_test_scaled = class_scaler.transform(
    X_class_test
)


logger.info(
    "Classification data prepared: "
    "%d training rows, %d test rows.",
    len(X_class_train),
    len(X_class_test)
)


print("Classification training shape:", X_class_train.shape)
print("Classification test shape:", X_class_test.shape)

print("\nTraining high-risk percentage:")
print(
    round(
        y_class_train.mean() * 100,
        2
    )
)

print("\nTest high-risk percentage:")
print(
    round(
        y_class_test.mean() * 100,
        2
    )
)

2026-10-07 20:11:52,533 - INFO - __main__ - Classification data prepared: 38549 training rows, 9638 test rows.


Classification training shape: (38549, 26)
Classification test shape: (9638, 26)

Training high-risk percentage:
11.28

Test high-risk percentage:
11.29


## 1.8 Classification Models

Two classification algorithms are trained and compared.

### Logistic Regression

Logistic Regression provides a simple and interpretable baseline model.

### Random Forest Classifier

Random Forest provides a non-linear ensemble model capable of learning more complex relationships between traffic, time and weather features.

Because the proxy high-risk target is imbalanced, `class_weight="balanced"` is used for both models.

In [23]:
# ------------------------------------------------------------
# Logistic Regression
# ------------------------------------------------------------

logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(
    X_class_train_scaled,
    y_class_train
)

logger.info(
    "Logistic Regression classification model trained successfully."
)

2026-10-07 20:12:06,822 - INFO - __main__ - Logistic Regression classification model trained successfully.


In [24]:
# ------------------------------------------------------------
# Random Forest Classifier
# ------------------------------------------------------------

rf_classifier = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_classifier.fit(
    X_class_train,
    y_class_train
)

logger.info(
    "Random Forest classification model trained successfully."
)

2026-10-07 20:12:09,640 - INFO - __main__ - Random Forest classification model trained successfully.


## 1.9 Classification Model Evaluation

The two classification models are evaluated on the same held-out test dataset.

The required evaluation metrics are:

- Accuracy
- Precision
- Recall
- F1-score
- ROC AUC

Because the proxy high-risk target is imbalanced, accuracy alone may be misleading. Precision, recall, F1-score and ROC AUC are therefore considered alongside accuracy when comparing the models.

In [25]:
# ------------------------------------------------------------
# Logistic Regression predictions
# ------------------------------------------------------------

logistic_pred = logistic_model.predict(
    X_class_test_scaled
)

logistic_prob = logistic_model.predict_proba(
    X_class_test_scaled
)[:, 1]


# ------------------------------------------------------------
# Random Forest predictions
# ------------------------------------------------------------

rf_pred = rf_classifier.predict(
    X_class_test
)

rf_prob = rf_classifier.predict_proba(
    X_class_test
)[:, 1]


# ------------------------------------------------------------
# Evaluation function
# ------------------------------------------------------------

def classification_metrics(
    y_true,
    y_pred,
    y_prob
):
    return {
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_true,
            y_prob
        )
    }


# ------------------------------------------------------------
# Compare models
# ------------------------------------------------------------

classification_results = pd.DataFrame(
    {
        "Logistic Regression":
            classification_metrics(
                y_class_test,
                logistic_pred,
                logistic_prob
            ),

        "Random Forest":
            classification_metrics(
                y_class_test,
                rf_pred,
                rf_prob
            )
    }
).T


logger.info(
    "Classification models evaluated successfully."
)

classification_results.round(4)

2026-10-07 20:12:10,905 - INFO - __main__ - Classification models evaluated successfully.


,Accuracy,Precision,Recall,F1,ROC_AUC
Logistic Regression,0.9656,0.7747,0.9798,0.8653,0.9937
Random Forest,0.9851,0.9076,0.9660,0.9359,0.9966



## 1.10 Regression Models

The second supervised learning task predicts `traffic_volume`.

Two regression algorithms are compared:

- Linear Regression as a simple baseline
- Random Forest Regressor as a non-linear ensemble model

The same engineered time, weather and holiday-related predictor set is used. `traffic_volume` is used only as the regression target.

The models are evaluated using Mean Absolute Error (MAE) and R-squared (R²), as required by the assignment.

In [32]:
# ------------------------------------------------------------
# Regression dataset
# ------------------------------------------------------------

X_reg = df[feature_columns].copy()
y_reg = df["traffic_volume"].copy()


# ------------------------------------------------------------
# Train/test split
# ------------------------------------------------------------

(
    X_reg_train,
    X_reg_test,
    y_reg_train,
    y_reg_test
) = train_test_split(
    X_reg,
    y_reg,
    test_size=0.20,
    random_state=42
)


# ------------------------------------------------------------
# Scaling for Linear Regression
# ------------------------------------------------------------

reg_scaler = StandardScaler()

X_reg_train_scaled = reg_scaler.fit_transform(
    X_reg_train
)

X_reg_test_scaled = reg_scaler.transform(
    X_reg_test
)


logger.info(
    "Regression data prepared: %d training rows, %d test rows.",
    len(X_reg_train),
    len(X_reg_test)
)

print("Regression training shape:", X_reg_train.shape)
print("Regression test shape:", X_reg_test.shape)

2026-10-07 20:36:05,171 - INFO - __main__ - Regression data prepared: 38549 training rows, 9638 test rows.


Regression training shape: (38549, 26)
Regression test shape: (9638, 26)


In [33]:
# ------------------------------------------------------------
# Linear Regression
# ------------------------------------------------------------

linear_model = LinearRegression()

linear_model.fit(
    X_reg_train_scaled,
    y_reg_train
)

logger.info(
    "Linear Regression model trained successfully."
)

2026-10-07 20:36:14,974 - INFO - __main__ - Linear Regression model trained successfully.


In [34]:
# ------------------------------------------------------------
# Random Forest Regressor
# ------------------------------------------------------------

rf_regressor = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_regressor.fit(
    X_reg_train,
    y_reg_train
)

logger.info(
    "Random Forest Regression model trained successfully."
)

2026-10-07 20:36:41,201 - INFO - __main__ - Random Forest Regression model trained successfully.


## 1.11 Regression Model Evaluation

The regression models are evaluated on the same held-out test dataset.

Two required metrics are used:

- **Mean Absolute Error (MAE)** – the average absolute difference between predicted and actual traffic volume. Lower values indicate better predictions.
- **R-squared (R²)** – the proportion of variation in traffic volume explained by the model. Higher values indicate better predictive performance.

The two models are compared using both metrics rather than relying on a single performance measure.

In [35]:
# ------------------------------------------------------------
# Task 1.11 - Regression model evaluation
# ------------------------------------------------------------

# Predictions
linear_pred = linear_model.predict(
    X_reg_test_scaled
)

rf_reg_pred = rf_regressor.predict(
    X_reg_test
)


# ------------------------------------------------------------
# Calculate required regression metrics
# ------------------------------------------------------------

regression_results = pd.DataFrame(
    {
        "MAE": [
            mean_absolute_error(
                y_reg_test,
                linear_pred
            ),
            mean_absolute_error(
                y_reg_test,
                rf_reg_pred
            )
        ],

        "R_squared": [
            r2_score(
                y_reg_test,
                linear_pred
            ),
            r2_score(
                y_reg_test,
                rf_reg_pred
            )
        ]
    },
    index=[
        "Linear Regression",
        "Random Forest"
    ]
)


logger.info(
    "Regression models evaluated successfully."
)


# User-facing model comparison
regression_results.round(4)

2026-10-07 20:39:26,831 - INFO - __main__ - Regression models evaluated successfully.


,MAE,R_squared
Linear Regression,813.7666,0.7259
Random Forest,271.2124,0.9442


## 1.12 Model Comparison, Findings and Limitations

### Classification

Two algorithms were evaluated for the proxy accident-risk classification task using the Part 3 quartile-based congestion definition.

The Logistic Regression model achieved an accuracy of **0.9656**, precision of **0.7747**, recall of **0.9798**, F1-score of **0.8653**, and ROC AUC of **0.9937**.

The model achieved particularly high recall, indicating that it identified most proxy high-risk observations. However, its lower precision indicates more false-positive high-risk classifications than the Random Forest model.

The Random Forest Classifier achieved an accuracy of **0.9851**, precision of **0.9076**, recall of **0.9660**, F1-score of **0.9359**, and ROC AUC of **0.9966**.

Overall, the Random Forest Classifier provides the stronger balance between precision and recall and is selected as the preferred classification model. Logistic Regression may still be useful in a safety-oriented scenario where maximising recall is considered more important than reducing false-positive alerts.

### Part 2 and Part 3 Congestion Definitions

The Part 2 congestion categories matched the Part 3 quartile definition for only **50.01%** of records. This difference occurred because Part 2 used fixed traffic-volume thresholds, whereas Part 3 explicitly requires congestion categories derived from quartiles of `traffic_volume`.

For Part 3, the congestion thresholds were therefore recalculated using the traffic-volume quartiles:

- Q1 = **1,192.50**
- Q2 = **3,379.00**
- Q3 = **4,933.00**

The `congestion_category` was then rebuilt using these quartile thresholds before the proxy `high_risk` classification target was created.

This ensures that the classification workflow follows the proxy-label definition specified in the Part 3 assignment.

### Classification Limitation

The classification results should still be interpreted cautiously.

The `high_risk` target is a proxy rather than an observed accident outcome. In the resulting dataset, **5,438 observations (11.29%)** were classified as proxy high risk.

The proxy is defined from a combination of High/Severe congestion and severe or low-visibility weather conditions. Weather-related variables also remain among the model predictors.

Therefore, part of the strong classification performance reflects the model learning variables that contribute to the construction of the proxy label rather than discovering an independent signal of actual accident risk.

For this reason, the reported classification metrics demonstrate the machine learning classification workflow but should not be interpreted as evidence of real-world accident-prediction performance.

### Regression

The Linear Regression baseline achieved an MAE of **813.77 vehicles** and an R-squared value of **0.7259**.

The Random Forest Regressor achieved an MAE of **271.21 vehicles** and an R-squared value of **0.9442**.

The Random Forest Regressor therefore produced substantially lower prediction error and explained a greater proportion of variation in traffic volume. It is selected as the preferred regression model.

These results suggest that non-linear relationships between time, weather and traffic conditions are important for predicting traffic volume.

### Regression Limitation

The regression models were evaluated using a random train/test split.

Because the traffic observations are sequential, a random split can place observations from nearby time periods in both the training and test datasets. Consequently, the Random Forest R-squared value of **0.9442** may be optimistic compared with performance on a completely future, unseen time period.

A production-oriented evaluation should therefore also use a chronological or time-based holdout set to assess future generalisation.



In [36]:
# ------------------------------------------------------------
# Task 1.13 - Final validation
# ------------------------------------------------------------

required_classification_metrics = {
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC"
}

required_regression_metrics = {
    "MAE",
    "R_squared"
}


# ------------------------------------------------------------
# Validate required evaluation metrics
# ------------------------------------------------------------

classification_metrics_present = (
    required_classification_metrics
    .issubset(classification_results.columns)
)

regression_metrics_present = (
    required_regression_metrics
    .issubset(regression_results.columns)
)


# ------------------------------------------------------------
# Validate required models
# ------------------------------------------------------------

classification_models_complete = (
    len(classification_results) >= 2
)

regression_models_complete = (
    len(regression_results) >= 2
)


# ------------------------------------------------------------
# Validate proxy classification target
# ------------------------------------------------------------

proxy_target_complete = (
    "high_risk" in df.columns
    and df["high_risk"].nunique() == 2
)


# ------------------------------------------------------------
# Validate required engineered features
# ------------------------------------------------------------

required_engineered_features = [
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "is_holiday"
]

feature_set_complete = all(
    feature in feature_columns
    for feature in required_engineered_features
)


# ------------------------------------------------------------
# Validate Part 3 quartile congestion categories
# ------------------------------------------------------------

expected_congestion_categories = {
    "Low",
    "Medium",
    "High",
    "Severe"
}

actual_congestion_categories = set(
    df["congestion_category"]
    .dropna()
    .unique()
)

quartile_congestion_complete = (
    expected_congestion_categories
    == actual_congestion_categories
)


# ------------------------------------------------------------
# Validate Part 3 quartile thresholds
# ------------------------------------------------------------

quartile_thresholds_complete = (
    q1 < q2 < q3
)


# ------------------------------------------------------------
# Final Task 1 validation
# ------------------------------------------------------------

task1_validation = {
    "Classification metrics complete":
        classification_metrics_present,

    "Two classification algorithms trained":
        classification_models_complete,

    "Regression metrics complete":
        regression_metrics_present,

    "Two regression algorithms trained":
        regression_models_complete,

    "Proxy classification target valid":
        proxy_target_complete,

    "Required engineered features included":
        feature_set_complete,

    "Part 3 quartile congestion categories created":
        quartile_congestion_complete,

    "Part 3 quartile thresholds valid":
        quartile_thresholds_complete
}


logger.info(
    "Task 1 final validation completed."
)


# ------------------------------------------------------------
# User-facing validation output
# ------------------------------------------------------------

print("TASK 1 FINAL VALIDATION")
print("-" * 60)

for check, passed in task1_validation.items():
    status = "PASS" if passed else "FAIL"
    print(f"{check}: {status}")


all_checks_passed = all(
    task1_validation.values()
)

print("-" * 60)

if all_checks_passed:
    print("TASK 1 STATUS: PASS - All required checks completed.")
else:
    print("TASK 1 STATUS: FAIL - Review the failed checks.")

2026-10-07 20:43:05,106 - INFO - __main__ - Task 1 final validation completed.


TASK 1 FINAL VALIDATION
------------------------------------------------------------
Classification metrics complete: PASS
Two classification algorithms trained: PASS
Regression metrics complete: PASS
Two regression algorithms trained: PASS
Proxy classification target valid: PASS
Required engineered features included: PASS
Part 3 quartile congestion categories created: PASS
Part 3 quartile thresholds valid: PASS
------------------------------------------------------------
TASK 1 STATUS: PASS - All required checks completed.


## 1.14 Task 1 Summary

Task 1 developed and evaluated four supervised machine learning models using the engineered traffic dataset from Part 2.

For classification, Logistic Regression and Random Forest were compared using accuracy, precision, recall, F1-score and ROC AUC. Random Forest provided the strongest overall balance of classification performance, while Logistic Regression achieved the highest recall.

For regression, Linear Regression and Random Forest Regression were compared using MAE and R-squared. Random Forest Regression substantially outperformed the linear baseline.

The supervised learning workflow uses a common engineered feature set containing time, weather, holiday and cyclical features.`traffic_volume` and `congestion_category` were excluded from the classification predictors, although weather features still contribute to the proxy label. 

Because no real accident dataset was supplied, the classification target is a documented proxy and should not be interpreted as an actual accident prediction.